# 03 - Évaluation du checkpoint SFT sur le jeu de test

## Objectif

Ce notebook évalue le checkpoint SFT LoRA entraîné pendant 3 époques sur le jeu `sft_test.jsonl`.

Le jeu de test n'a pas été utilisé pendant l'entraînement.  
Il permet de mesurer la capacité du modèle à généraliser sur des exemples médicaux séparés du train et de la validation.

L'évaluation porte sur :
- la test loss ;
- quelques générations qualitatives ;
- les limites observées avant l'étape DPO.

In [1]:
import torch
import pandas as pd

from datasets import load_dataset
from transformers import (
    AutoTokenizer,
    AutoModelForCausalLM,
    Trainer,
    TrainingArguments,
    DataCollatorForSeq2Seq,
)
from peft import PeftModel

d:\Documents\Formation IA\Projet 14\projet14_311\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
BASE_MODEL_NAME = "Qwen/Qwen3-1.7B-Base"
SFT_CHECKPOINT_DIR = "../../outputs/qwen3-sft-3epochs/checkpoint-4762"

DATA_DIR = "../../data"
TEST_FILE = f"{DATA_DIR}/sft_test.jsonl"

MAX_LENGTH = 1024
SEED = 42

## Chargement du tokenizer

Le tokenizer sauvegardé avec le checkpoint SFT est chargé afin d'utiliser la même configuration que pendant l'entraînement.

In [3]:
tokenizer = AutoTokenizer.from_pretrained(
    SFT_CHECKPOINT_DIR,
    trust_remote_code=True
)

if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

print("Tokenizer chargé")
print("Pad token :", tokenizer.pad_token)
print("EOS token :", tokenizer.eos_token)

Tokenizer chargé
Pad token : <|endoftext|>
EOS token : <|endoftext|>


## Chargement du modèle SFT LoRA

Le modèle de base `Qwen/Qwen3-1.7B-Base` est chargé, puis les poids LoRA du checkpoint SFT 3 époques sont appliqués.

Le modèle est placé en mode évaluation.

In [4]:
device = "cuda" if torch.cuda.is_available() else "cpu"

base_model = AutoModelForCausalLM.from_pretrained(
    BASE_MODEL_NAME,
    dtype=torch.float16 if torch.cuda.is_available() else torch.float32,
    trust_remote_code=True,
    low_cpu_mem_usage=False,
)

model = PeftModel.from_pretrained(
    base_model,
    SFT_CHECKPOINT_DIR
)

model.to(device)
model.eval()

print("Modèle SFT chargé")
print("Device principal :", next(model.parameters()).device)

Loading weights: 100%|██████████| 310/310 [00:02<00:00, 144.24it/s]


Modèle SFT chargé
Device principal : cuda:0


## Chargement du jeu de test SFT

Le fichier `sft_test.jsonl` est chargé pour évaluer le checkpoint SFT sur des exemples non utilisés pendant l'entraînement ni la validation.

In [5]:
test_dataset = load_dataset(
    "json",
    data_files=TEST_FILE,
    split="train"
)

print("Test :", len(test_dataset))
print(test_dataset[0])

Test : 576
{'instruction': "Un pédiatre vous appelle pour un nourrisson de 3 mois hospitalisé pour insuffisance cardiaque. Il est rose et ses pouls sont tous perçus. On ausculte un petit souffle éjectionnel sur la voie pulmonaire. La radiographie de thorax montre un cœur en dextrocardie avec une hypoplasie importante du poumon droit et une surcharge vasculaire modérée du poumon gauche. Il vous décrit le compte-rendu d'échographie: CIA ostium secundum large shuntant gauche-droite, cavités droites dilatées, HTAP à 67 mm Hg, cavités gauches normales, APG bien vue, APD non vue, flux continu anormal dans le foie.\n\nQuestion : Quel diagnostic évoquez-vous?", 'response': 'Syndrome du cimeterre', 'source': 'MediQAl', 'language': 'fr'}


## Préparation du jeu de test

Le jeu de test est converti au même format conversationnel que celui utilisé pendant l'entraînement.

Le prompt utilisateur est masqué dans les labels afin que la test loss soit calculée uniquement sur la réponse assistant attendue.

In [6]:
def build_prompt(example):
    return (
        "<|im_start|>user\n"
        + example["instruction"].strip()
        + "\n<|im_end|>\n"
        + "<|im_start|>assistant\n"
    )

def build_full_text(example):
    return (
        build_prompt(example)
        + example["response"].strip()
        + "\n<|im_end|>"
    )

def tokenize_function(example):
    prompt_text = build_prompt(example)
    full_text = build_full_text(example)

    tokenized_full = tokenizer(
        full_text,
        truncation=True,
        max_length=MAX_LENGTH,
        add_special_tokens=False,
    )

    tokenized_prompt = tokenizer(
        prompt_text,
        truncation=True,
        max_length=MAX_LENGTH,
        add_special_tokens=False,
    )

    input_ids = tokenized_full["input_ids"]
    attention_mask = tokenized_full["attention_mask"]

    labels = input_ids.copy()

    prompt_length = len(tokenized_prompt["input_ids"])
    prompt_length = min(prompt_length, len(labels))

    labels[:prompt_length] = [-100] * prompt_length

    return {
        "input_ids": input_ids,
        "attention_mask": attention_mask,
        "labels": labels,
    }

## Tokenisation du jeu de test

La tokenisation est appliquée au jeu de test.  
Les exemples ne contenant aucun label entraînable après troncature sont retirés afin de garantir un calcul valide de la test loss.

In [7]:
tokenized_test = test_dataset.map(
    tokenize_function,
    remove_columns=test_dataset.column_names
)

def has_trainable_labels(example):
    return any(label != -100 for label in example["labels"])

print("Avant filtrage test :", len(tokenized_test))

tokenized_test = tokenized_test.filter(has_trainable_labels)

print("Après filtrage test :", len(tokenized_test))
print(tokenized_test)

Map: 100%|██████████| 576/576 [00:00<00:00, 1096.21 examples/s]


Avant filtrage test : 576


Filter: 100%|██████████| 576/576 [00:00<00:00, 4186.64 examples/s]

Après filtrage test : 565
Dataset({
    features: ['input_ids', 'attention_mask', 'labels'],
    num_rows: 565
})


## Data collator

Le data collator applique le padding dynamique pendant l'évaluation.

In [8]:
data_collator = DataCollatorForSeq2Seq(
    tokenizer=tokenizer,
    padding=True
)

print("Data collator prêt")

Data collator prêt


## Évaluation sur le jeu de test

Le checkpoint SFT 3 époques est évalué sur le jeu `sft_test.jsonl`.

La métrique principale est la `test_loss`, calculée uniquement sur les tokens de réponse assistant.

In [9]:
eval_args = TrainingArguments(
    output_dir="../../outputs/sft_test_eval",
    per_device_eval_batch_size=1,
    fp16=torch.cuda.is_available(),
    report_to="none",
    remove_unused_columns=False,
)

trainer = Trainer(
    model=model,
    args=eval_args,
    eval_dataset=tokenized_test,
    data_collator=data_collator,
)

test_metrics = trainer.evaluate()
test_metrics

Training Loss,Validation Loss,Step
No log,2.236565,0


{'eval_loss': 2.236565351486206}

In [ ]:
print("Test loss :", test_metrics["eval_loss"])
print("Nombre d'exemples test évalués :", len(tokenized_test))

## Sélection du checkpoint SFT

Deux checkpoints du run SFT complet ont été évalués sur le jeu `sft_test.jsonl`.

| Checkpoint | Epoch | Test loss |
|---|---:|---:|
| `checkpoint-4762` | 2 | 2.2366 |
| checkpoint final | 3 | 3.2027 |

Le checkpoint de l'époque 2 obtient une test loss nettement plus faible que le checkpoint final de l'époque 3.

Ce résultat suggère que l'entraînement supplémentaire à l'époque 3 a amélioré la train loss, mais a dégradé la généralisation sur le jeu de test.  
Le checkpoint `checkpoint-4762` est donc retenu comme meilleur checkpoint SFT pour la suite du projet.

Ce choix illustre l'importance de ne pas sélectionner automatiquement le dernier checkpoint, mais de comparer les performances sur un jeu de test séparé.